# Project Summary

This individual capstone develops an IT support chatbot for common workplace issues such as email, VPN, Teams, printers, and account access.

The project uses 94 training examples, 11 validation examples, and 11 held-out test examples. Qwen2.5-0.5B-Instruct was fine-tuned with LoRA for three epochs in Google Colab. A Gradio interface allows users to ask questions live.

On the 11 test questions, manual usefulness scores improved from 6/22 for the base model to 11/22 for the fine-tuned model. The bot still gives some incomplete or incorrect advice, so its answers require review.

GitHub repository:[GitHub Link](https://github.com/jaindolly296/it-support-chatbot)

## 1. Load training data

This code downloads 96 IT support question-answer examples. The model will learn from these examples during training.

In [1]:
import json
from urllib.request import urlopen

url = "https://huggingface.co/datasets/w1z4rd3k/it-support-l1-ticket-classification/resolve/main/data/en/train.jsonl"

with urlopen(url) as response:
    train_records = [
        json.loads(line)
        for line in response.read().decode("utf-8").splitlines()
        if line.strip()
    ]

print("Training examples:", len(train_records))
print("First question:", train_records[0]["user_message"])
print("First answer:", train_records[0]["expected_response"])

Training examples: 96
First question: I can sign in to Windows, but the company portal says my account is locked after a few failed attempts yesterday.
First answer: Your account looks locked. Wait 15 minutes, try signing in once with the correct password, and let us know the exact sign-in error if it still fails.


## 2. Load validation and test data

This code loads separate examples for checking the model. Validation data helps us monitor training; test data is used for the final evaluation.

In [2]:
base_url = "https://huggingface.co/datasets/w1z4rd3k/it-support-l1-ticket-classification/resolve/main/data/en/"

def load_records(filename):
    with urlopen(base_url + filename) as response:
        return [
            json.loads(line)
            for line in response.read().decode("utf-8").splitlines()
            if line.strip()
        ]

validation_records = load_records("validation.jsonl")
test_records = load_records("test.jsonl")

print("Validation examples:", len(validation_records))
print("Test examples:", len(test_records))

Validation examples: 12
Test examples: 12


## 3. Check dataset quality

This step counts missing question-answer fields and repeated questions before training.

In [3]:
from collections import Counter

all_records = train_records + validation_records + test_records
required_fields = ("user_message", "expected_response", "category")

missing_rows = sum(
    any(not str(row.get(field, "")).strip() for field in required_fields)
    for row in all_records
)

questions = [
    row["user_message"].strip().casefold()
    for row in all_records
]
duplicate_questions = len(questions) - len(set(questions))

print("Total examples:", len(all_records))
print("Rows with missing fields:", missing_rows)
print("Repeated questions:", duplicate_questions)
print("Categories:", Counter(row["category"] for row in all_records))

Total examples: 120
Rows with missing fields: 0
Repeated questions: 0
Categories: Counter({'email_issue': 10, 'wifi_issue': 9, 'account_access': 8, 'password_reset': 8, 'vpn_issue': 8, 'printer_issue': 8, 'network_issue': 8, 'software_issue': 8, 'hardware_issue': 8, 'mfa_issue': 8, 'browser_issue': 8, 'performance_issue': 8, 'shared_drive_access': 7, 'teams_issue': 7, 'device_setup': 7})


## 4. Review sample answers

This step displays a few training examples so I can check whether the questions and answers are clear and useful.

In [4]:
import random

sample_rows = random.Random(42).sample(train_records, 5)

for number, row in enumerate(sample_rows, start=1):
    print(f"\nExample {number}")
    print("Category:", row["category"])
    print("Question:", row["user_message"])
    print("Answer:", row["expected_response"])


Example 1
Category: email_issue
Question: Outlook is warning that my mailbox is almost full, and I am worried new mail will stop arriving.
Answer: Your mailbox is close to its storage limit. Delete unneeded mail, empty Deleted Items, and archive older messages if your team uses archiving.

Example 2
Category: email_issue
Question: Sent mail is staying queued since this morning, and recipients say they never got anything from me.
Answer: Check Outlook connection status, confirm the mailbox is not full, and test sending a simple message without attachments.

Example 3
Category: account_access
Question: I am setting up a replacement laptop and the office sign-in page says this account has been blocked.
Answer: Your account looks locked. Wait 15 minutes, try signing in once with the correct password, and let us know the exact sign-in error if it still fails.

Example 4
Category: email_issue
Question: Outlook on my iPhone keeps asking me to re-enter the account password every few hours.
An

## 5. Data review observations

I reviewed five sample answers. Most give useful first-level IT support guidance. However, the account-lock answer assumes a 15-minute waiting period, which may differ between organizations. The mobile Outlook answer may also need organization-specific guidance. These examples require review before training.

The public dataset contains synthetic IT support examples. My four manually written examples have links to official Microsoft support pages. I will report these two data sources separately.

## 6. Remove questionable answers

Some synthetic answers assume organization-specific rules. This step excludes the two answer patterns identified during manual review.

In [5]:
questionable_phrases = (
    "wait 15 minutes",
    "re-add the work account",
)

def clean_records(records):
    return [
        row for row in records
        if all(
            str(row.get(field, "")).strip()
            for field in ("user_message", "expected_response", "category")
        )
        and not any(
            phrase in row["expected_response"].lower()
            for phrase in questionable_phrases
        )
    ]

clean_train = clean_records(train_records)
clean_validation = clean_records(validation_records)
clean_test = clean_records(test_records)

print("Removed from training:", len(train_records) - len(clean_train))
print("Removed from validation:", len(validation_records) - len(clean_validation))
print("Removed from test:", len(test_records) - len(clean_test))

Removed from training: 6
Removed from validation: 1
Removed from test: 1


## 7. Verify the cleaned data splits

This step checks the remaining example counts and ensures that identical questions do not appear in both training and evaluation data.

In [6]:
def question_set(records):
    return {row["user_message"].strip().casefold() for row in records}

train_questions = question_set(clean_train)
validation_questions = question_set(clean_validation)
test_questions = question_set(clean_test)

print("Clean training examples:", len(clean_train))
print("Clean validation examples:", len(clean_validation))
print("Clean test examples:", len(clean_test))
print("Train/validation overlaps:", len(train_questions & validation_questions))
print("Train/test overlaps:", len(train_questions & test_questions))
print("Validation/test overlaps:", len(validation_questions & test_questions))

Clean training examples: 90
Clean validation examples: 11
Clean test examples: 11
Train/validation overlaps: 0
Train/test overlaps: 0
Validation/test overlaps: 0


## 8. Save the cleaned dataset

This step saves separate training, validation, and test files in Google Drive so the experiment can be repeated later.

In [7]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

data_folder = Path(
    "/content/drive/MyDrive/IT_Support_Chatbot_Project/data"
)
data_folder.mkdir(parents=True, exist_ok=True)

for filename, records in {
    "train_clean.jsonl": clean_train,
    "validation_clean.jsonl": clean_validation,
    "test_clean.jsonl": clean_test,
}.items():
    with (data_folder / filename).open("w", encoding="utf-8") as file:
        for row in records:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved files in:", data_folder)

Mounted at /content/drive
Saved files in: /content/drive/MyDrive/IT_Support_Chatbot_Project/data


## 9. Load verified examples

This step reads the four manually prepared examples and checks their questions, answers, categories, and source links.

In [10]:
import csv

csv_path = data_folder / "verified_examples.csv"

with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
    verified_examples = list(csv.DictReader(file))

print("Verified examples:", len(verified_examples))
print("Columns:", list(verified_examples[0].keys()))

for row in verified_examples:
    print(row["category"], "—", row["question"])

Verified examples: 4
Columns: ['question', 'answer', 'category', 'source_url']
Wi-Fi and internet — My Windows laptop is connected to Wi-Fi, but the internet is not working. What should I check?
Login and password — I forgot my Microsoft account password. How can I reset it?
Software installation — I cannot install an app from Microsoft Store on my Windows PC. What should I try?
Basic troubleshooting — My Windows PC is running slowly. What should I check first?


## 10. Add verified examples to training data

This step adds four source-linked IT support examples to the cleaned training data. Validation and test data remain separate.

In [11]:
manual_records = []

for number, row in enumerate(verified_examples, start=1):
    manual_records.append({
        "id": f"manual-{number:03d}",
        "language": "en",
        "user_message": row["question"].strip(),
        "expected_response": row["answer"].strip(),
        "category": row["category"].strip(),
        "source": "official_support_paraphrase",
        "source_url": row["source_url"].strip(),
    })

final_train = clean_train + manual_records

print("Public training examples:", len(clean_train))
print("Verified examples added:", len(manual_records))
print("Final training examples:", len(final_train))

Public training examples: 90
Verified examples added: 4
Final training examples: 94


## 11. Check the final training data

This step checks that the added questions are complete and do not duplicate training or evaluation questions.

In [12]:
final_questions = [
    row["user_message"].strip().casefold()
    for row in final_train
]

missing_final = sum(
    not row["user_message"].strip() or not row["expected_response"].strip()
    for row in final_train
)

print("Missing questions or answers:", missing_final)
print("Repeated training questions:", len(final_questions) - len(set(final_questions)))
print("Overlap with validation:", len(set(final_questions) & validation_questions))
print("Overlap with test:", len(set(final_questions) & test_questions))
print("Verified source links:", sum(bool(row["source_url"]) for row in manual_records))

Missing questions or answers: 0
Repeated training questions: 0
Overlap with validation: 0
Overlap with test: 0
Verified source links: 4


## 12. Save the final training data

This step saves the combined training examples as a separate file. The original cleaned files are kept for comparison.

In [13]:
final_path = data_folder / "train_final.jsonl"

with final_path.open("w", encoding="utf-8") as file:
    for row in final_train:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", final_path)
print("Examples saved:", len(final_train))

Saved: /content/drive/MyDrive/IT_Support_Chatbot_Project/data/train_final.jsonl
Examples saved: 94


## 13. Check the GPU

This step checks whether Colab has a GPU available for model training.

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


## 14. Install model training libraries

These libraries are needed to load a pretrained model, prepare the dataset, and fine-tune the model.

In [2]:
%pip -q install -U transformers peft accelerate datasets
print("Training libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.0 MB/s eta 0:00:00
Training libraries installed.


## 15. Reload the saved dataset

This step loads the final training, validation, and test files from Google Drive after the runtime change.

In [1]:
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

data_folder = Path("/content/drive/MyDrive/IT_Support_Chatbot_Project/data")

def read_jsonl(filename):
    with (data_folder / filename).open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

final_train = read_jsonl("train_final.jsonl")
clean_validation = read_jsonl("validation_clean.jsonl")
clean_test = read_jsonl("test_clean.jsonl")

print("Training:", len(final_train))
print("Validation:", len(clean_validation))
print("Test:", len(clean_test))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training: 94
Validation: 11
Test: 11


## 16. Load the pretrained model

This project uses Qwen2.5-0.5B-Instruct as the starting model. Its answers before training will serve as the baseline.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("Model loaded:", MODEL_ID)
print("Device:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0


## 17. Test one baseline answer

This step asks the original model one validation question before fine-tuning.

In [3]:
SYSTEM_PROMPT = (
    "You are an IT helpdesk assistant. Give concise, practical steps. "
    "If important details are missing, ask a clarifying question."
)

def generate_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

question = clean_validation[0]["user_message"]
print("Question:", question)
print("Original model answer:", generate_answer(question))

Question: My Android mail app keeps asking for updated credentials after my password expired overnight.
Original model answer: To resolve the issue where your Android email app is prompting you to update your credentials after your password expires, follow these steps:

1. **Check Your Email Settings:**
   - Open your email account settings.
   - Look for any options related to "Password" or "Security."
   - Ensure that your password has been changed and saved.

2. **Reset Password:**
   - Go back to your account settings.
   - Click on "Forgot Password?"
   - Enter your current password and click "Change Password."

3. **Update Credentials:**
   - After resetting your password, go back to your email account settings again.


## 18. Save baseline validation answers

This step records the original model's answers to all validation questions so they can be compared with the fine-tuned model later.

In [6]:
results_folder = data_folder.parent / "results"
results_folder.mkdir(parents=True, exist_ok=True)

baseline_rows = []

for row in clean_validation:
    baseline_rows.append({
        "id": row["id"],
        "category": row["category"],
        "question": row["user_message"],
        "expected_answer": row["expected_response"],
        "base_answer": generate_answer(row["user_message"]),
    })

baseline_path = results_folder / "baseline_validation.jsonl"

with baseline_path.open("w", encoding="utf-8") as file:
    for row in baseline_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Baseline answers saved:", len(baseline_rows))
print("File:", baseline_path)

Baseline answers saved: 11
File: /content/drive/MyDrive/IT_Support_Chatbot_Project/results/baseline_validation.jsonl


## 19. Prepare question-answer examples for training

This step converts each question and answer into the model's chat format. Only answer tokens will be used to calculate training loss.

In [4]:
def encode_record(row):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["user_message"]},
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    answer_ids = tokenizer(
        row["expected_response"].strip() + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    return {
        "input_ids": prompt_ids + answer_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(answer_ids)),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

train_encoded = [encode_record(row) for row in final_train]
validation_encoded = [encode_record(row) for row in clean_validation]

print("Encoded training examples:", len(train_encoded))
print("Encoded validation examples:", len(validation_encoded))

Encoded training examples: 94
Encoded validation examples: 11


## 20. Check token lengths

This step checks that the formatted examples fit within the chosen training length and contain answer tokens.

In [5]:
MAX_LENGTH = 512

all_encoded = train_encoded + validation_encoded
lengths = [len(row["input_ids"]) for row in all_encoded]
answer_lengths = [
    sum(label != -100 for label in row["labels"])
    for row in all_encoded
]

print("Longest example tokens:", max(lengths))
print("Examples over limit:", sum(length > MAX_LENGTH for length in lengths))
print("Shortest answer tokens:", min(answer_lengths))

Longest example tokens: 130
Examples over limit: 0
Shortest answer tokens: 19


## 21. Add a LoRA adapter

The LoRA adapter makes a small part of the model trainable while keeping the pretrained model weights frozen.

## Fix incompatible audio library

This text chatbot does not use torchaudio. Removing its incompatible version allows the LoRA setup to run.

In [9]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 22. Create training batches

This step turns the encoded examples into datasets and pads each batch so examples of different lengths can be trained together.

In [7]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

train_dataset = Dataset.from_list(train_encoded)
validation_dataset = Dataset.from_list(validation_encoded)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_model.config.pad_token_id = tokenizer.pad_token_id

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

print("Training rows:", len(train_dataset))
print("Validation rows:", len(validation_dataset))

Training rows: 94
Validation rows: 11


## 23. Set training parameters

The adapter will train for three epochs. Validation loss will be checked after each epoch, and the best checkpoint will be selected.

In [8]:
from transformers import Trainer, TrainingArguments

model_folder = data_folder.parent / "model"
model_folder.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(model_folder / "checkpoints"),
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    seed=42,
    label_names=["labels"],
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
)

print("Trainer ready.")
print("Epochs:", training_args.num_train_epochs)

Trainer ready.
Epochs: 3


## 24. Fine-tune and save the model

This step trains the LoRA adapter, checks validation loss after each epoch, and saves the best adapter and training log to Google Drive.

In [11]:
train_output = trainer.train()

adapter_folder = model_folder / "it_support_lora_adapter"
peft_model.save_pretrained(str(adapter_folder))
tokenizer.save_pretrained(str(adapter_folder))

log_path = results_folder / "training_log.json"
with log_path.open("w", encoding="utf-8") as file:
    json.dump(trainer.state.log_history, file, indent=2)

print("Training completed.")
print("Epochs completed:", trainer.state.epoch)
print("Best validation loss:", trainer.state.best_metric)
print("Adapter saved:", adapter_folder)
print("Training log saved:", log_path)

Epoch,Training Loss,Validation Loss
1,2.001316,1.851899
2,1.793717,1.708015
3,1.668537,1.654661


NameError: name 'results_folder' is not defined

In [12]:
results_folder = data_folder.parent / "results"
results_folder.mkdir(parents=True, exist_ok=True)

## 25. Save training results

Training completed for three epochs. This step saves its recorded losses without running training again.

In [13]:
results_folder = data_folder.parent / "results"
results_folder.mkdir(parents=True, exist_ok=True)

log_path = results_folder / "training_log.json"
with log_path.open("w", encoding="utf-8") as file:
    json.dump(trainer.state.log_history, file, indent=2)

print("Training log saved:", log_path)
print("Epochs completed:", trainer.state.epoch)
print("Best validation loss:", trainer.state.best_metric)

Training log saved: /content/drive/MyDrive/IT_Support_Chatbot_Project/results/training_log.json
Epochs completed: 3.0
Best validation loss: 1.654660701751709


## 26. Verify the saved adapter

This step checks that the trained LoRA adapter files exist in Google Drive.

In [14]:
adapter_folder = data_folder.parent / "model" / "it_support_lora_adapter"

print("Adapter config exists:", (adapter_folder / "adapter_config.json").exists())
print("Adapter weights exist:", (adapter_folder / "adapter_model.safetensors").exists())
print("Tokenizer files exist:", (adapter_folder / "tokenizer_config.json").exists())

Adapter config exists: True
Adapter weights exist: True
Tokenizer files exist: True


## 27. Compare one answer before and after training

This step asks the fine-tuned model a validation question and compares it with the saved baseline answer.

In [15]:
peft_model.eval()

def generate_finetuned_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(next(peft_model.parameters()).device)

    with torch.no_grad():
        output = peft_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

baseline_path = results_folder / "baseline_validation.jsonl"
with baseline_path.open("r", encoding="utf-8") as file:
    first_baseline = json.loads(next(file))

question = first_baseline["question"]
print("Question:", question)
print("Expected answer:", first_baseline["expected_answer"])
print("Before training:", first_baseline["base_answer"])
print("After training:", generate_finetuned_answer(question))

Question: My Android mail app keeps asking for updated credentials after my password expired overnight.
Expected answer: Please reset your password through the approved reset page, then sign out and back in to affected apps after a short wait for sync.
Before training: To resolve the issue where your Android email app is prompting you to update your credentials after your password expires, follow these steps:

1. **Check Your Email Settings:**
   - Open your email account settings.
   - Look for any options related to "Password" or "Security."
   - Ensure that your password has been changed and saved.

2. **Reset Password:**
   - Go back to your account settings.
   - Click on "Forgot Password?"
   - Enter your current password and click "Change Password."

3. **Update Credentials:**
   - After resetting your password, go back to your email account settings again.
After training: This is normal, and the app should automatically update your account with new passwords once it has been au

## 28. Verify training losses

This step checks the saved validation losses and identifies the best recorded epoch.

In [16]:
with log_path.open("r", encoding="utf-8") as file:
    saved_history = json.load(file)

validation_losses = [
    (entry["epoch"], entry["eval_loss"])
    for entry in saved_history
    if "eval_loss" in entry
]

print("Validation losses by epoch:", validation_losses)
print("Lowest logged loss:", min(loss for _, loss in validation_losses))
print("Trainer best loss:", trainer.state.best_metric)

Validation losses by epoch: [(1.0, 1.851899266242981), (2.0, 1.708014965057373), (3.0, 1.654660701751709)]
Lowest logged loss: 1.654660701751709
Trainer best loss: 1.654660701751709


## 29. Reload the saved model

This step loads the adapter from Google Drive to confirm that the saved model can be used after this Colab session ends.

In [17]:
from peft import PeftModel

fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

saved_model = PeftModel.from_pretrained(
    fresh_base,
    str(adapter_folder),
)
saved_model.eval()

print("Saved adapter loaded successfully.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Saved adapter loaded successfully.


## 30. Check saved-model consistency

This step checks whether the model loaded from Drive gives the same answer as the trained model in memory.

In [18]:
def answer_with_model(model_to_use, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(next(model_to_use.parameters()).device)

    with torch.no_grad():
        output = model_to_use.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

check_question = clean_validation[0]["user_message"]
current_answer = answer_with_model(peft_model, check_question)
saved_answer = answer_with_model(saved_model, check_question)

print("Saved and current answers match:", saved_answer == current_answer)

Saved and current answers match: True


## 31. Save fine-tuned validation answers

This step records the saved model's answers to the same validation questions used for the baseline.

In [19]:
validation_after = []

for row in clean_validation:
    validation_after.append({
        "id": row["id"],
        "category": row["category"],
        "question": row["user_message"],
        "expected_answer": row["expected_response"],
        "fine_tuned_answer": answer_with_model(
            saved_model, row["user_message"]
        ),
    })

validation_after_path = results_folder / "finetuned_validation.jsonl"
with validation_after_path.open("w", encoding="utf-8") as file:
    for row in validation_after:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Fine-tuned validation answers saved:", len(validation_after))

Fine-tuned validation answers saved: 11


## 32. Review validation answers

This step shows three questions with the expected answer, original model answer, and fine-tuned model answer for manual review.

In [20]:
with baseline_path.open("r", encoding="utf-8") as file:
    baseline_saved = [json.loads(line) for line in file if line.strip()]

baseline_by_id = {row["id"]: row for row in baseline_saved}

for row in validation_after[:3]:
    print("\nQUESTION:", row["question"])
    print("EXPECTED:", row["expected_answer"])
    print("BEFORE:", baseline_by_id[row["id"]]["base_answer"])
    print("AFTER:", row["fine_tuned_answer"])


QUESTION: My Android mail app keeps asking for updated credentials after my password expired overnight.
EXPECTED: Please reset your password through the approved reset page, then sign out and back in to affected apps after a short wait for sync.
BEFORE: To resolve the issue where your Android email app is prompting you to update your credentials after your password expires, follow these steps:

1. **Check Your Email Settings:**
   - Open your email account settings.
   - Look for any options related to "Password" or "Security."
   - Ensure that your password has been changed and saved.

2. **Reset Password:**
   - Go back to your account settings.
   - Click on "Forgot Password?"
   - Enter your current password and click "Change Password."

3. **Update Credentials:**
   - After resetting your password, go back to your email account settings again.
AFTER: This is normal, and the app should automatically update your account with new passwords once it has been authenticated. Restart the

## 33. Run the held-out test

These questions were not used for training. This step records both the original and fine-tuned model's answers for final evaluation.

In [21]:
test_comparisons = []

for row in clean_test:
    question = row["user_message"]

    with saved_model.disable_adapter():
        base_answer = answer_with_model(saved_model, question)

    tuned_answer = answer_with_model(saved_model, question)

    test_comparisons.append({
        "id": row["id"],
        "category": row["category"],
        "question": question,
        "expected_answer": row["expected_response"],
        "base_answer": base_answer,
        "fine_tuned_answer": tuned_answer,
    })

test_path = results_folder / "test_comparison.jsonl"
with test_path.open("w", encoding="utf-8") as file:
    for row in test_comparisons:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Final test questions evaluated:", len(test_comparisons))
print("Saved:", test_path)

Final test questions evaluated: 11
Saved: /content/drive/MyDrive/IT_Support_Chatbot_Project/results/test_comparison.jsonl


## 34. Prepare manual test review

This step creates a table containing all 11 test questions and both model answers so their usefulness can be scored manually.

In [22]:
import csv

review_path = results_folder / "test_review_template.csv"
columns = [
    "id", "category", "question", "expected_answer",
    "base_answer", "fine_tuned_answer",
    "base_score", "fine_tuned_score", "notes"
]

with review_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    for row in test_comparisons:
        writer.writerow({
            **row,
            "base_score": "",
            "fine_tuned_score": "",
            "notes": "",
        })

print("Rows ready for review:", len(test_comparisons))
print("Saved:", review_path)

Rows ready for review: 11
Saved: /content/drive/MyDrive/IT_Support_Chatbot_Project/results/test_review_template.csv


# Human Evaluation of Test Answers
Each answer was scored from 0 to 2: 0 = incorrect, 1 = partially useful, 2 = correct and useful.

In [23]:
import csv

score_file = results_folder / "test_review_scored.csv"

with score_file.open(encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file))

assert len(rows) == 11
base_scores = [int(row["base_score"]) for row in rows]
tuned_scores = [int(row["fine_tuned_score"]) for row in rows]
assert all(score in (0, 1, 2) for score in base_scores + tuned_scores)

maximum = 2 * len(rows)
print(f"Test questions: {len(rows)}")
print(f"Base model: {sum(base_scores)}/{maximum} ({sum(base_scores)/maximum:.1%})")
print(f"Fine-tuned model: {sum(tuned_scores)}/{maximum} ({sum(tuned_scores)/maximum:.1%})")

Test questions: 11
Base model: 6/22 (27.3%)
Fine-tuned model: 11/22 (50.0%)


# Evaluation Results and Limitations
On 11 held-out IT support questions, the base model scored 6/22 and the fine-tuned model scored 11/22 using a manual 0–2 usefulness scale. Fine-tuning improved the overall score, but several answers remained incomplete or inaccurate. This is a small, subjective evaluation, so the score should not be presented as accuracy. More verified training examples and broader testing are needed before practical deployment.

# Interactive IT Support Chatbot UI
Users type IT support questions in a chat window and receive answers from the fine-tuned model.

# IT Support Chatbot with Suggested Questions
The sidebar helps users choose common IT support questions quickly.

In [35]:
import gradio as gr

suggested_questions = [
    "My Outlook emails are stuck in Outbox. What should I check?",
    "I changed my password and VPN will not connect. What can I try?",
    "Teams is using the wrong microphone. How do I change it?",
    "My office printer shows offline. What should I check?",
    "I cannot access a shared folder. What should I do?",
    "Wi-Fi is connected but websites will not load. What can I try?"
]

def respond(message, history):
    message = message.strip()
    if not message:
        return "", history or []

    answer = answer_with_model(saved_model, message)
    history = (history or []) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return "", history

if "demo" in globals():
    demo.close()

with gr.Blocks(title="IT Support Chatbot") as demo:
    gr.Markdown("# IT Support Chatbot\n### Developed by Dolly Jain")

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown("### Suggested questions\nClick a question to use it.")
            buttons = [gr.Button(q) for q in suggested_questions]

        with gr.Column(scale=3, min_width=600):
            chatbot = gr.Chatbot(height=460)

            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Type your IT support question...",
                    show_label=False,
                    scale=6
                )
                send_button = gr.Button("Send", variant="primary", scale=1)

            clear_button = gr.Button("Clear chat")

    for question, button in zip(suggested_questions, buttons):
        button.click(
            lambda q=question: q,
            inputs=[],
            outputs=question_box
        )

    question_box.submit(
        respond,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    )
    send_button.click(
        respond,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    )
    clear_button.click(
        lambda: ("", []),
        inputs=[],
        outputs=[question_box, chatbot]
    )

demo.launch(inline=True)

Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f967e0e945dcd6f0c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Live Demo Observations
The email answer was weak: checking push notifications does not diagnose why outgoing mail fails. The VPN answer gave basic retry steps but omitted password and MFA checks. The Teams answer suggested useful headset checks but omitted the microphone selection in Teams settings. These examples show that the bot can respond to new questions, but its advice still needs human review.

# Improvement Plan
Add more verified IT support examples for email, VPN, and Teams troubleshooting. Check each answer against approved support guidance, then evaluate the improved bot on new questions that were not used for training. For uncertain cases, the bot should recommend contacting IT support instead of inventing settings or steps.